In [1]:
!pip install transformers[torch] datasets scikit-learn evaluate torch torchvision torchaudio

In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import classification_report
import numpy as np
import evaluate
import re

c:\Users\Lenovo\Documents\GitHub\cs180-nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# load the datasets
data = {
    'train': load_dataset("json", data_files="../data/train.json1", split="train"),
    'dev': load_dataset("csv", data_files="../data/dev.csv", split="train")
}

# preprocess data
def clean_text(s: str):
    # Replace all curly double quotes with "
    s = re.sub(r'[“”]', '"', s)
     
    # Replace all curly single quotes with '
    s = re.sub(r"[‘’]", "'", s)
    
    # Replace en dash and em dash with hyphen
    s = re.sub(r"[–—]", "-", s)

    # Only retain alphanumeric, whitespace characters, single and double quotes, and hyphens
    s = re.sub(pattern=rf"[^a-zA-Z0-9\s\-\'\"]", repl="", string=s, flags=re.IGNORECASE)

    # Remove extra whitespaces
    s = re.sub(pattern=r"\s+", repl=" ", string=s).strip()

    return s

def preprocess(text: str):
    return clean_text(text)

# load tokenizer of base bert model 
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# tokenize and preprocess the datasets
def tokenize_function(examples):
    cleaned_text = [preprocess(text) for text in examples["text"]]
    return tokenizer(cleaned_text, padding="max_length", truncation=True)

tokenized_datasets = {
    split: dataset.map(tokenize_function, batched=True)
    for split, dataset in data.items()
}

In [4]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=3)

# Training arguments
training_args = TrainingArguments(
    fp16=True,
    num_train_epochs=8, 
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    learning_rate=3e-5,
    weight_decay=0.01,
    lr_scheduler_type="inverse_sqrt",
    warmup_steps=200
)

# performance metrics
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=predictions, references=labels, average="macro")["precision"],
        "recall": recall_metric.compute(predictions=predictions, references=labels, average="macro")["recall"],
        "f1": f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"],
    }

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# Train
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Evaluate
eval_results = trainer.evaluate()
predictions_output = trainer.predict(tokenized_datasets["dev"])
logits = predictions_output.predictions
true_labels = predictions_output.label_ids
predicted_labels = np.argmax(logits, axis=1)

# Results
print(classification_report(true_labels, predicted_labels, digits=4))
print("Evaluation Results:", eval_results)

# Save Model
trainer.save_model("../models/bert")
tokenizer.save_pretrained("../models/bert")  

              precision    recall  f1-score   support

           0     0.7903    0.9245    0.8522        53
           1     0.8611    0.7654    0.8105        81
           2     0.7308    0.7308    0.7308        26

    accuracy                         0.8125       160
   macro avg     0.7941    0.8069    0.7978       160
weighted avg     0.8165    0.8125    0.8113       160

Evaluation Results: {'eval_loss': 1.256119728088379, 'eval_accuracy': 0.8125, 'eval_precision': 0.7940676408418343, 'eval_recall': 0.8069098771404851, 'eval_f1': 0.7978002200508594, 'eval_runtime': 1.0291, 'eval_samples_per_second': 155.481, 'eval_steps_per_second': 19.435, 'epoch': 8.0}


('./models/bert\\tokenizer_config.json',
 './models/bert\\special_tokens_map.json',
 './models/bert\\vocab.txt',
 './models/bert\\added_tokens.json',
 './models/bert\\tokenizer.json')